In [4]:
# code_1 — Reading RAW JSON from OneLake (creating df) 
from pyspark.sql.functions import col, explode, avg, min, max, length, upper

raw_path = "Files/raw/exchange_rates/date=*/base=*/*.json"

df = (
    spark.read
         .option("multiline", True)
         .json(raw_path)
)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 6, Finished, Available, Finished)

In [5]:
# code_2 — Listing folders in OneLake 
from notebookutils import mssparkutils

mssparkutils.fs.ls("Files/")


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 7, Finished, Available, Finished)

[FileInfo(path=abfss://b5d25385-7e76-4ef6-af76-780e4546f67c@onelake.dfs.fabric.microsoft.com/59a838a2-d927-45f0-ad67-e6ad478b18db/Files/raw, name=raw, size=0)]

In [6]:
# code_3 — Re-reading RAW JSON (duplicate step) 
from pyspark.sql.functions import col, explode, avg, min, max, length, upper

raw_path = "Files/raw/exchange_rates/date=*/base=*/*.json"

df = (
    spark.read
         .option("multiline", True)
         .json(raw_path)
)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 8, Finished, Available, Finished)

In [18]:
# code_13 — Clean data → create clean df 
clean = (
    flat
    .filter(
        col("BaseCurrency").isNotNull()
        & col("TargetCurrency").isNotNull()
        & col("ExchangeRate").isNotNull()
    )
    .withColumn("BaseCurrency", upper(col("BaseCurrency")))
    .withColumn("TargetCurrency", upper(col("TargetCurrency")))
    .filter(length(col("TargetCurrency")) == 3)
    .withColumn("ExchangeRate", col("ExchangeRate").cast("decimal(18,6)"))
    .withColumn("RateDate", col("RateDate").cast("date"))
)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 20, Finished, Available, Finished)

In [26]:
# code_21 — Create exchange_rates (Curated layer) 
clean.createOrReplaceTempView("vw_clean")
spark.sql("""
CREATE OR REPLACE TABLE exchange_rates
USING DELTA
AS
SELECT *
FROM vw_clean
""")


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 28, Finished, Available, Finished)

DataFrame[]

In [4]:
# code_22 — Create exchange_rates_stats (Curated layer)
from pyspark.sql.functions import avg, min, max

# Якщо stats ще не визначений — обчислюємо його
if "stats" not in locals():
    stats = (
        clean
        .groupBy("TargetCurrency")
        .agg(
            avg("ExchangeRate").alias("AvgRate"),
            min("ExchangeRate").alias("MinRate"),
            max("ExchangeRate").alias("MaxRate")
        )
    )

stats.createOrReplaceTempView("vw_stats")

spark.sql("""
CREATE OR REPLACE TABLE exchange_rates_stats
USING DELTA
AS
SELECT *
FROM vw_stats
""")






StatementMeta(, 56d60d1a-8883-49a6-9aeb-2f928911604e, 6, Finished, Available, Finished)

NameError: name 'clean' is not defined

In [25]:
# code_20 — Show stats (3 rows) 
stats.show(3, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 27, Finished, Available, Finished)

+--------------+------------+--------+--------+
|TargetCurrency|AvgRate     |MinRate |MaxRate |
+--------------+------------+--------+--------+
|DKK           |7.2356750000|7.232600|7.244900|
|MYR           |4.4962500000|4.493000|4.506000|
|NZD           |1.7857750000|1.784300|1.790200|
+--------------+------------+--------+--------+
only showing top 3 rows



In [24]:
# code_19 — Show clean (3 rows) 
clean.show(3, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 26, Finished, Available, Finished)

+------------+----------+--------------+------------+
|BaseCurrency|RateDate  |TargetCurrency|ExchangeRate|
+------------+----------+--------------+------------+
|USD         |2025-01-20|AUD           |1.611800    |
|USD         |2025-01-20|BGN           |1.895900    |
|USD         |2025-01-20|BRL           |6.076800    |
+------------+----------+--------------+------------+
only showing top 3 rows



In [23]:
# code_18 — Print schema of stats 
stats.printSchema()


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 25, Finished, Available, Finished)

root
 |-- TargetCurrency: string (nullable = false)
 |-- AvgRate: decimal(22,10) (nullable = true)
 |-- MinRate: decimal(18,6) (nullable = true)
 |-- MaxRate: decimal(18,6) (nullable = true)



In [22]:
# code_17 — Display stats (Fabric UI) 
display(stats)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 24, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 69ce4c6d-8eda-42af-82fe-c0bbbc53e9a8)

In [21]:
# code_16 — Aggregate statistics per TargetCurrency 
stats = (
    clean
    .groupBy("TargetCurrency")
    .agg(
        avg("ExchangeRate").alias("AvgRate"),
        min("ExchangeRate").alias("MinRate"),
        max("ExchangeRate").alias("MaxRate")
    )
)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 23, Finished, Available, Finished)

In [20]:
# code_15 — Print schema of clean 
clean.printSchema()


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 22, Finished, Available, Finished)

root
 |-- BaseCurrency: string (nullable = true)
 |-- RateDate: date (nullable = true)
 |-- TargetCurrency: string (nullable = false)
 |-- ExchangeRate: decimal(18,6) (nullable = true)



In [19]:
# code_14 — Show clean (20 rows) 
clean.show(20, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 21, Finished, Available, Finished)

+------------+----------+--------------+------------+
|BaseCurrency|RateDate  |TargetCurrency|ExchangeRate|
+------------+----------+--------------+------------+
|USD         |2025-01-20|AUD           |1.611800    |
|USD         |2025-01-20|BGN           |1.895900    |
|USD         |2025-01-20|BRL           |6.076800    |
|USD         |2025-01-20|CAD           |1.446700    |
|USD         |2025-01-20|CHF           |0.914020    |
|USD         |2025-01-20|CNY           |7.312000    |
|USD         |2025-01-20|CZK           |24.437000   |
|USD         |2025-01-20|DKK           |7.232600    |
|USD         |2025-01-20|EUR           |0.969370    |
|USD         |2025-01-20|GBP           |0.819970    |
|USD         |2025-01-20|HKD           |7.784100    |
|USD         |2025-01-20|HUF           |400.490000  |
|USD         |2025-01-20|IDR           |16390.000000|
|USD         |2025-01-20|ILS           |3.586100    |
|USD         |2025-01-20|INR           |86.540000   |
|USD         |2025-01-20|ISK

In [16]:
# code_12 — Show first 5 rows of flat 
flat.show(5, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 18, Finished, Available, Finished)

+------------+----------+--------------+------------+
|BaseCurrency|RateDate  |TargetCurrency|ExchangeRate|
+------------+----------+--------------+------------+
|USD         |2025-01-20|AUD           |1.6118      |
|USD         |2025-01-20|BGN           |1.8959      |
|USD         |2025-01-20|BRL           |6.0768      |
|USD         |2025-01-20|CAD           |1.4467      |
|USD         |2025-01-20|CHF           |0.91402     |
+------------+----------+--------------+------------+
only showing top 5 rows



In [15]:
# code_11 — Print schema of flat 
flat.printSchema()


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 17, Finished, Available, Finished)

root
 |-- BaseCurrency: string (nullable = true)
 |-- RateDate: string (nullable = true)
 |-- TargetCurrency: string (nullable = false)
 |-- ExchangeRate: double (nullable = true)



In [14]:
# code_10 — Show flat (20 rows) 
flat.show(20, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 16, Finished, Available, Finished)

+------------+----------+--------------+------------+
|BaseCurrency|RateDate  |TargetCurrency|ExchangeRate|
+------------+----------+--------------+------------+
|USD         |2025-01-20|AUD           |1.6118      |
|USD         |2025-01-20|BGN           |1.8959      |
|USD         |2025-01-20|BRL           |6.0768      |
|USD         |2025-01-20|CAD           |1.4467      |
|USD         |2025-01-20|CHF           |0.91402     |
|USD         |2025-01-20|CNY           |7.312       |
|USD         |2025-01-20|CZK           |24.437      |
|USD         |2025-01-20|DKK           |7.2326      |
|USD         |2025-01-20|EUR           |0.96937     |
|USD         |2025-01-20|GBP           |0.81997     |
|USD         |2025-01-20|HKD           |7.7841      |
|USD         |2025-01-20|HUF           |400.49      |
|USD         |2025-01-20|IDR           |16390.0     |
|USD         |2025-01-20|ILS           |3.5861      |
|USD         |2025-01-20|INR           |86.54       |
|USD         |2025-01-20|ISK

In [13]:
# code_9 — Flatten rates → explode → create flat 
expr_str = "map(" + ", ".join([f"'{c}', rates.`{c}`" for c in rates_cols]) + ")"

flat = df.selectExpr(
    "base as BaseCurrency",
    "date as RateDate",
    f"explode({expr_str}) as (TargetCurrency, ExchangeRate)"
)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 15, Finished, Available, Finished)

In [12]:
# code_8 — Extract currency list from rates 
rates_cols = df.select("rates.*").columns
rates_cols


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 14, Finished, Available, Finished)

['AUD',
 'BGN',
 'BRL',
 'CAD',
 'CHF',
 'CNY',
 'CZK',
 'DKK',
 'EUR',
 'GBP',
 'HKD',
 'HUF',
 'IDR',
 'ILS',
 'INR',
 'ISK',
 'JPY',
 'KRW',
 'MXN',
 'MYR',
 'NOK',
 'NZD',
 'PHP',
 'PLN',
 'RON',
 'SEK',
 'SGD',
 'THB',
 'TRY',
 'ZAR']

In [11]:
# code_7 — Print df schema (again) 
df.printSchema()


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 13, Finished, Available, Finished)

root
 |-- amount: double (nullable = true)
 |-- base: string (nullable = true)
 |-- date: string (nullable = true)
 |-- rates: struct (nullable = true)
 |    |-- AUD: double (nullable = true)
 |    |-- BGN: double (nullable = true)
 |    |-- BRL: double (nullable = true)
 |    |-- CAD: double (nullable = true)
 |    |-- CHF: double (nullable = true)
 |    |-- CNY: double (nullable = true)
 |    |-- CZK: double (nullable = true)
 |    |-- DKK: double (nullable = true)
 |    |-- EUR: double (nullable = true)
 |    |-- GBP: double (nullable = true)
 |    |-- HKD: double (nullable = true)
 |    |-- HUF: double (nullable = true)
 |    |-- IDR: long (nullable = true)
 |    |-- ILS: double (nullable = true)
 |    |-- INR: double (nullable = true)
 |    |-- ISK: double (nullable = true)
 |    |-- JPY: double (nullable = true)
 |    |-- KRW: double (nullable = true)
 |    |-- MXN: double (nullable = true)
 |    |-- MYR: double (nullable = true)
 |    |-- NOK: double (nullable = true)
 |    |-- 

In [10]:
# code_6 — Display df (Fabric UI) 
display(df)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3a2d090d-3c05-462c-bb9d-7fa5c1fe9a1a)

In [8]:
# code_5 — Show 5 rows of df 
df.show(5, truncate=False)


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 10, Finished, Available, Finished)

+------+----+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|amount|base|date      |rates                                                                                                                                                                                                                                             |
+------+----+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1.0   |USD |2025-01-20|{1.6118, 1.8959, 6.0768, 1.4467, 0.91402, 7.312, 24.437, 7.2326, 0.96937, 0.81997, 7.7841, 400.49, 16390, 3.5861, 86.54, 141.04, 156.42, 1451.34, 20.87, 4.493, 11.4061, 1.7

In [7]:
# code_4 — Print schema of df 
df.printSchema()


StatementMeta(, d18d9e47-40d9-4a00-9a4d-d66b1c34b0e7, 9, Finished, Available, Finished)

root
 |-- amount: double (nullable = true)
 |-- base: string (nullable = true)
 |-- date: string (nullable = true)
 |-- rates: struct (nullable = true)
 |    |-- AUD: double (nullable = true)
 |    |-- BGN: double (nullable = true)
 |    |-- BRL: double (nullable = true)
 |    |-- CAD: double (nullable = true)
 |    |-- CHF: double (nullable = true)
 |    |-- CNY: double (nullable = true)
 |    |-- CZK: double (nullable = true)
 |    |-- DKK: double (nullable = true)
 |    |-- EUR: double (nullable = true)
 |    |-- GBP: double (nullable = true)
 |    |-- HKD: double (nullable = true)
 |    |-- HUF: double (nullable = true)
 |    |-- IDR: long (nullable = true)
 |    |-- ILS: double (nullable = true)
 |    |-- INR: double (nullable = true)
 |    |-- ISK: double (nullable = true)
 |    |-- JPY: double (nullable = true)
 |    |-- KRW: double (nullable = true)
 |    |-- MXN: double (nullable = true)
 |    |-- MYR: double (nullable = true)
 |    |-- NOK: double (nullable = true)
 |    |-- 